# Gold | Construção das fatos (em `gold_staging`)

Medidas de resultado, metas e diferenças no grão indicado em cada fato. `situacao_meta` é uma classificação derivada
(estados em `dim_situacao_meta`). **Ausência de meta nunca vira meta zero.**

**Recortes comparáveis (correção):**
- Meta municipal só existe para a rede municipal (código `'3'`).
- Meta estadual é um agregado da rede **Pública**; por isso `ft_meta_vs_resultado_uf` compara **apenas resultados da rede pública**
  (identificada pelo dicionário oficial, sem código fixo). Antes, resultados de todas as redes eram comparados à meta pública.

**Taxa ponderada:** `ft_taxa_alfabetizacao_ponderada` recalcula a taxa dos microdados como
`soma(peso dos alfabetizados) / soma(peso dos alunos avaliados)`, para conferência com a taxa oficial (notebook de validação cruzada).

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
dbutils.widgets.text("codigo_presente", "1", "[CONFIRMAR no dicionário] código de presenca = aluno avaliado")
dbutils.widgets.text("codigo_alfabetizado", "1", "[CONFIRMAR no dicionário] código de alfabetizado = alfabetizado")
_m = iniciar_etapa()
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold_staging")

In [0]:
%sql
-- Resultado municipal: uma linha por ano, município, série e rede.
CREATE OR REPLACE TABLE workspace.gold_staging.ft_resultado_municipio USING DELTA AS
SELECT ano, id_municipio, serie, rede_codigo, taxa_alfabetizacao, media_portugues,
       proporcao_aluno_nivel_0, proporcao_aluno_nivel_1, proporcao_aluno_nivel_2,
       proporcao_aluno_nivel_3, proporcao_aluno_nivel_4, proporcao_aluno_nivel_5,
       proporcao_aluno_nivel_6, proporcao_aluno_nivel_7, proporcao_aluno_nivel_8
FROM workspace.silver.municipio;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_staging.ft_resultado_uf USING DELTA AS
SELECT ano, sigla_uf, serie, rede_codigo, taxa_alfabetizacao, media_portugues,
       proporcao_aluno_nivel_0, proporcao_aluno_nivel_1, proporcao_aluno_nivel_2,
       proporcao_aluno_nivel_3, proporcao_aluno_nivel_4, proporcao_aluno_nivel_5,
       proporcao_aluno_nivel_6, proporcao_aluno_nivel_7, proporcao_aluno_nivel_8
FROM workspace.silver.uf;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_staging.ft_resultado_brasil USING DELTA AS
SELECT ano, rede_codigo, taxa_alfabetizacao, percentual_participacao
FROM workspace.silver.meta_alfabetizacao_brasil;

In [0]:
%sql
-- Colunas anuais de metas -> um registro por ano-meta (mantém a trajetória mais recente por chave).
CREATE OR REPLACE TABLE workspace.gold_staging.ft_meta_brasil USING DELTA AS
SELECT ano AS ano_referencia, rede_codigo, ano_meta, meta_alfabetizacao
FROM workspace.silver.meta_alfabetizacao_brasil
UNPIVOT (meta_alfabetizacao FOR ano_meta IN
  (meta_alfabetizacao_2024 AS `2024`, meta_alfabetizacao_2025 AS `2025`, meta_alfabetizacao_2026 AS `2026`,
   meta_alfabetizacao_2027 AS `2027`, meta_alfabetizacao_2028 AS `2028`, meta_alfabetizacao_2029 AS `2029`, meta_alfabetizacao_2030 AS `2030`))
QUALIFY ROW_NUMBER() OVER (PARTITION BY rede_codigo, ano_meta ORDER BY ano_referencia DESC)=1;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_staging.ft_meta_uf USING DELTA AS
SELECT ano AS ano_referencia, sigla_uf, rede_codigo, ano_meta, meta_alfabetizacao
FROM workspace.silver.meta_alfabetizacao_uf
UNPIVOT (meta_alfabetizacao FOR ano_meta IN
  (meta_alfabetizacao_2024 AS `2024`, meta_alfabetizacao_2025 AS `2025`, meta_alfabetizacao_2026 AS `2026`,
   meta_alfabetizacao_2027 AS `2027`, meta_alfabetizacao_2028 AS `2028`, meta_alfabetizacao_2029 AS `2029`, meta_alfabetizacao_2030 AS `2030`))
QUALIFY ROW_NUMBER() OVER (PARTITION BY sigla_uf, rede_codigo, ano_meta ORDER BY ano_referencia DESC)=1;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_staging.ft_meta_municipio USING DELTA AS
SELECT ano AS ano_referencia, id_municipio, rede_codigo, ano_meta, meta_alfabetizacao
FROM workspace.silver.meta_alfabetizacao_municipio
UNPIVOT (meta_alfabetizacao FOR ano_meta IN
  (meta_alfabetizacao_2024 AS `2024`, meta_alfabetizacao_2025 AS `2025`, meta_alfabetizacao_2026 AS `2026`,
   meta_alfabetizacao_2027 AS `2027`, meta_alfabetizacao_2028 AS `2028`, meta_alfabetizacao_2029 AS `2029`, meta_alfabetizacao_2030 AS `2030`))
QUALIFY ROW_NUMBER() OVER (PARTITION BY id_municipio, rede_codigo, ano_meta ORDER BY ano_referencia DESC)=1;

In [0]:
%sql
-- Município x meta (somente rede municipal '3'). Trajetória = ano_referencia mais recente.
-- SEM_META_DISPONIVEL_PARA_COMPARACAO = ano do resultado fora da trajetória (LEFT JOIN, sem subconsulta correlacionada).
CREATE OR REPLACE TABLE workspace.gold_staging.ft_meta_vs_resultado_municipio USING DELTA AS
WITH metas_trajetoria AS (
    SELECT m.id_municipio, m.rede_codigo, CAST(m.ano_meta AS INT) AS ano, m.meta_alfabetizacao
    FROM workspace.gold_staging.ft_meta_municipio m
    WHERE m.ano_referencia = (SELECT MAX(ano_referencia) FROM workspace.gold_staging.ft_meta_municipio)
),
anos_com_meta AS (SELECT DISTINCT ano FROM metas_trajetoria)
SELECT r.ano, r.id_municipio, r.serie, r.rede_codigo, r.taxa_alfabetizacao,
       m.meta_alfabetizacao,
       r.taxa_alfabetizacao - m.meta_alfabetizacao AS diferenca_meta,
       CASE
        WHEN am.ano IS NULL THEN 'SEM_META_DISPONIVEL_PARA_COMPARACAO'
        WHEN m.meta_alfabetizacao IS NULL THEN 'SEM_META'
        WHEN r.taxa_alfabetizacao IS NULL THEN 'SEM_RESULTADO'
        WHEN r.taxa_alfabetizacao >= m.meta_alfabetizacao THEN 'ATINGIU_OU_SUPEROU'
        ELSE 'ABAIXO_DA_META'
    END AS situacao_meta
FROM workspace.gold_staging.ft_resultado_municipio r
LEFT JOIN metas_trajetoria m
       ON r.id_municipio = m.id_municipio AND r.rede_codigo = m.rede_codigo AND r.ano = m.ano
LEFT JOIN anos_com_meta am ON am.ano = r.ano
WHERE r.rede_codigo = '3';

In [0]:
%sql
-- UF x meta: a meta estadual é agregado da rede PÚBLICA -> só resultados da rede pública entram na comparação.
-- Redes não públicas ficam em ft_resultado_uf, mas fora desta comparação.
CREATE OR REPLACE TABLE workspace.gold_staging.ft_meta_vs_resultado_uf USING DELTA AS
WITH rede_publica AS (
    SELECT DISTINCT CAST(chave AS STRING) AS rede_codigo
    FROM workspace.silver.dicionario
    WHERE nome_coluna = 'rede' AND TRIM(valor) LIKE 'Pública%'
),
metas_trajetoria AS (
    SELECT m.sigla_uf, CAST(m.ano_meta AS INT) AS ano, m.meta_alfabetizacao
    FROM workspace.gold_staging.ft_meta_uf m
    WHERE m.ano_referencia = (SELECT MAX(ano_referencia) FROM workspace.gold_staging.ft_meta_uf)
),
anos_com_meta AS (SELECT DISTINCT ano FROM metas_trajetoria)
SELECT r.ano, r.sigla_uf, r.serie, r.rede_codigo, r.taxa_alfabetizacao,
       m.meta_alfabetizacao,
       r.taxa_alfabetizacao - m.meta_alfabetizacao AS diferenca_meta,
       CASE
        WHEN am.ano IS NULL THEN 'SEM_META_DISPONIVEL_PARA_COMPARACAO'
        WHEN m.meta_alfabetizacao IS NULL THEN 'SEM_META'
        WHEN r.taxa_alfabetizacao IS NULL THEN 'SEM_RESULTADO'
        WHEN r.taxa_alfabetizacao >= m.meta_alfabetizacao THEN 'ATINGIU_OU_SUPEROU'
        ELSE 'ABAIXO_DA_META'
    END AS situacao_meta
FROM workspace.gold_staging.ft_resultado_uf r
INNER JOIN rede_publica p ON r.rede_codigo = p.rede_codigo
LEFT JOIN metas_trajetoria m ON r.sigla_uf = m.sigla_uf AND r.ano = m.ano
LEFT JOIN anos_com_meta am ON am.ano = r.ano;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_staging.ft_meta_vs_resultado_brasil USING DELTA AS
SELECT r.ano, r.rede_codigo, r.taxa_alfabetizacao, m.meta_alfabetizacao,
       r.taxa_alfabetizacao - m.meta_alfabetizacao AS diferenca_meta,
       CASE
           WHEN m.meta_alfabetizacao IS NULL THEN 'SEM_META'
           WHEN r.taxa_alfabetizacao IS NULL THEN 'SEM_RESULTADO'
           WHEN r.taxa_alfabetizacao >= m.meta_alfabetizacao THEN 'ATINGIU_OU_SUPEROU'
           ELSE 'ABAIXO_DA_META'
       END AS situacao_meta
FROM workspace.gold_staging.ft_resultado_brasil r
LEFT JOIN workspace.gold_staging.ft_meta_brasil m
       ON r.ano = CAST(m.ano_meta AS INT) AND r.rede_codigo = m.rede_codigo;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_staging.ft_resultado_aluno_municipio USING DELTA AS
SELECT ano, id_municipio, serie, rede_codigo,
       alfabetizado AS categoria_alfabetizacao,
       COUNT(*) AS quantidade_alunos,
       AVG(proficiencia) AS proficiencia_media
FROM workspace.silver.alunos
GROUP BY ano, id_municipio, serie, rede_codigo, alfabetizado;

In [0]:
# Taxa ponderada dos microdados: soma(peso alfabetizados) / soma(peso avaliados).
# Denominador = apenas alunos avaliados (presença confirmada) com peso; NULL/0 nunca vira taxa 0.
# ROUND(...,6): somas de ponto flutuante em ordens diferentes davam 100.0000000000001 (erro numérico, não dado inválido).
_pres = dbutils.widgets.get("codigo_presente")
_alf = dbutils.widgets.get("codigo_alfabetizado")
spark.sql(f"""
CREATE OR REPLACE TABLE workspace.gold_staging.ft_taxa_alfabetizacao_ponderada USING DELTA AS
SELECT ano, id_municipio, serie, rede_codigo,
       COUNT(*)                                                         AS alunos_registrados,
       SUM(CASE WHEN presenca = '{_pres}' AND peso_aluno IS NOT NULL THEN 1 ELSE 0 END) AS alunos_avaliados,
       SUM(CASE WHEN presenca = '{_pres}' THEN peso_aluno END)        AS peso_avaliados,
       SUM(CASE WHEN presenca = '{_pres}' AND alfabetizado = '{_alf}' THEN peso_aluno END) AS peso_alfabetizados,
       CASE WHEN SUM(CASE WHEN presenca = '{_pres}' THEN peso_aluno END) > 0
            THEN ROUND(100.0 * SUM(CASE WHEN presenca = '{_pres}' AND alfabetizado = '{_alf}' THEN peso_aluno END)
                       / SUM(CASE WHEN presenca = '{_pres}' THEN peso_aluno END), 6)
       END AS taxa_alfabetizacao_ponderada
FROM workspace.silver.alunos
GROUP BY ano, id_municipio, serie, rede_codigo
""")

In [0]:
for t in ["ft_resultado_municipio", "ft_resultado_uf", "ft_resultado_brasil", "ft_meta_brasil", "ft_meta_uf",
          "ft_meta_municipio", "ft_meta_vs_resultado_municipio", "ft_meta_vs_resultado_uf",
          "ft_meta_vs_resultado_brasil", "ft_resultado_aluno_municipio", "ft_taxa_alfabetizacao_ponderada"]:
    registrar_execucao("GOLD", t, _m, row_count=contar_tabela(f"workspace.gold_staging.{t}"))